# 🔬 Pulp Fibril Instance Segmentation — Kaggle Training
**Pipeline:** Real-ESRGAN → Swin-T → AG-UNet → Mask2Former → GNN Quantification

> Run all cells top to bottom. Training takes ~10–14 hours on T4/P100.
> Download `checkpoints/best_checkpoint.pth` after training completes.

## Cell 1: Install Dependencies

In [ ]:
# Install all required packages
import subprocess
import sys

packages = [
    'timm>=0.9.12',
    'albumentations>=1.3.1',
    'scikit-image>=0.21.0',
    'networkx>=3.1',
    'einops>=0.7.0',
    'scipy>=1.11.0',
    'tqdm',
    'opencv-python-headless',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# torch-geometric for GNN
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric',
])

print('✅ All packages installed')

## Cell 2: Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'Mixed precision: ENABLED (saves ~40% VRAM)')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {DEVICE}')

## Cell 3: Setup Paths

In [ ]:
import os, sys
from pathlib import Path

# ── CHANGE THIS TO YOUR KAGGLE DATASET PATH ──
# After uploading your project as a Kaggle Dataset, it will appear at:
# /kaggle/input/YOUR-DATASET-NAME/

DATASET_PATH = '/kaggle/input/fibrilsynth'  # <-- Update this!
PROJECT_ROOT = DATASET_PATH

# Add to Python path
sys.path.insert(0, PROJECT_ROOT)

# Output dir (writeable)
OUTPUT_DIR = Path('/kaggle/working')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

DATA_ROOT = Path(DATASET_PATH) / 'data' / 'synthetic'

print(f'Project: {PROJECT_ROOT}')
print(f'Data:    {DATA_ROOT}')
print(f'Output:  {OUTPUT_DIR}')
print(f'Files found: {len(list(Path(PROJECT_ROOT).rglob("*.py")))} .py files')

## Cell 4: Generate Synthetic Dataset
> If you already uploaded a pre-generated dataset, skip this cell.

In [ ]:
# Generate dataset directly if not pre-generated
# This generates 500 images → takes ~8-12 min on CPU

import json
annot_path = DATA_ROOT / 'annotations.json'

if annot_path.exists():
    with open(annot_path) as f:
        coco = json.load(f)
    print(f'✅ Dataset already exists: {len(coco["images"])} images, {len(coco["annotations"])} annotations')
else:
    print('Generating synthetic dataset (500 images)...')
    # Copy data to writable location first
    DATA_ROOT_WRITE = OUTPUT_DIR / 'data' / 'synthetic'
    
    os.chdir(PROJECT_ROOT)
    !python data_pipeline/generate_dataset.py \
        --n 500 \
        --output {DATA_ROOT_WRITE} \
        --size 512 \
        --seed 42
    DATA_ROOT = DATA_ROOT_WRITE

## Cell 5: Visualize Sample (Data Sanity Check)

In [ ]:
import cv2
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import random

with open(DATA_ROOT / 'annotations.json') as f:
    coco = json.load(f)

# Pick random sample
sample = random.choice(coco['images'])
img = cv2.imread(str(DATA_ROOT / 'images' / sample['file_name']), cv2.IMREAD_GRAYSCALE)

# Load masks for this image
masks_dir = DATA_ROOT / 'masks' / f"{sample['id']:05d}"
masks = []
if masks_dir.exists():
    for mf in sorted(masks_dir.iterdir()):
        m = cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)
        if m is not None:
            masks.append(m)

# Colorize
vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR).astype(np.float32)
colors = [(np.random.randint(80,255), np.random.randint(80,255), np.random.randint(80,255))
          for _ in masks]
for mask, color in zip(masks, colors):
    colored = np.zeros_like(vis); colored[mask>127] = color
    vis = cv2.addWeighted(vis, 1.0, colored, 0.5, 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"FibrilSynth Sample: {sample['file_name']} ({len(masks)} fibers)", fontsize=13)
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(vis.astype(np.uint8)[...,::-1]); axes[1].set_title('Instance Masks'); axes[1].axis('off')
plt.tight_layout(); plt.show()
print(f'✅ Dataset looks good! {len(coco["images"])} images total')

## Cell 6: Build Model (Verify It Loads)

In [ ]:
import os
os.chdir(PROJECT_ROOT)

from models.mask2former import build_model

MODEL_CONFIG = {
    'backbone':   'swin_tiny',
    'pretrained': True,          # Downloads ImageNet-1K weights
    'num_queries': 50,
    'hidden_dim': 256,
    'nheads': 8,
    'dec_layers': 6,
    'num_classes': 1,
}

model = build_model(MODEL_CONFIG, device=DEVICE)

# Quick forward pass test
dummy = torch.zeros(1, 1, 512, 512).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'✅ Model forward pass OK')
print(f'   pred_logits: {out["pred_logits"].shape}')  # (1, 50, 2)
print(f'   pred_masks:  {out["pred_masks"].shape}')   # (1, 50, 128, 128)

## Cell 7: TRAIN (Full Training Loop)
> ⚠️ This cell runs for ~10–14 hours. Make sure to enable **Save & Run All** in Kaggle settings.

In [ ]:
KAGGLE_CONFIG = {
    'data_root': str(DATA_ROOT),
    'image_size': 512,
    'batch_size': 1,
    'accum_steps': 8,
    'num_workers': 2,
    'epochs': 30,
    'warmup_epochs': 5,
    'backbone': 'swin_tiny',
    'pretrained': True,
    'num_queries': 50,
    'hidden_dim': 256,
    'nheads': 8,
    'dec_layers': 6,
    'num_classes': 1,
    'lr_backbone': 1e-5,
    'lr_head': 1e-4,
    'weight_decay': 0.05,
    'grad_clip': 0.01,
    'loss_weights': {'loss_ce': 2.0, 'loss_mask': 5.0, 'loss_dice': 5.0},
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'use_wandb': False,
    'device': DEVICE,
    'mixed_precision': True,
    'seed': 42,
}

from training.train import train
train(KAGGLE_CONFIG)

## Cell 8: Plot Training Curves

In [ ]:
import json, matplotlib.pyplot as plt

history_path = CHECKPOINT_DIR / 'training_history.json'
with open(history_path) as f:
    history = json.load(f)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train_loss'], label='Train Loss', color='#4ecdc4', linewidth=2)
ax.plot(history['val_loss'],   label='Val Loss',   color='#ff6b6b', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Progress', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150)
plt.show()
print(f"Best val loss: {min(history['val_loss']):.4f} at epoch {history['val_loss'].index(min(history['val_loss']))+1}")

## Cell 9: Quick Inference Demo (on 3 Test Images)

In [ ]:
import random
from inference.predict import predict

# Pick 3 random test images
img_dir = DATA_ROOT / 'images'
test_images = random.sample(list(img_dir.glob('*.png')), min(3, len(list(img_dir.glob('*.png')))))

for img_path in test_images:
    metrics = predict(
        image_path=str(img_path),
        checkpoint_path=str(CHECKPOINT_DIR / 'best_checkpoint.pth'),
        output_dir=str(OUTPUT_DIR / 'predictions'),
        px_to_um=0.25,
        use_sr=False,    # Skip SR in notebook (faster)
        device=DEVICE,
    )
    
    # Show the saved visualization
    vis_path = OUTPUT_DIR / 'predictions' / f"{img_path.stem}_segmented.png"
    if vis_path.exists():
        vis = cv2.imread(str(vis_path))
        plt.figure(figsize=(12, 5))
        plt.imshow(vis[...,::-1])
        plt.title(f'{img_path.name} — {len(metrics)} fibrils detected')
        plt.axis('off'); plt.show()

## Cell 10: Download Checkpoint
> After training, use the Kaggle output panel to download `best_checkpoint.pth`

In [ ]:
import os
ckpt = CHECKPOINT_DIR / 'best_checkpoint.pth'
if ckpt.exists():
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'✅ Checkpoint ready for download: {ckpt}')
    print(f'   Size: {size_mb:.1f} MB')
    print(f'   Go to: Kaggle Notebook → Output → Download best_checkpoint.pth')
else:
    print('❌ Checkpoint not found — did training complete?')